# Анализ отзывов покупателей

**Данные:** отзывы с Яндекс.Карт, собранные вручную с публичных страниц 5 московских баблти-кафе.

**Почему вручную?** Яндекс.Карты загружают тексты отзывов через JavaScript: тест подтвердил, что в статическом HTML тексты отзывов отсутствуют (слово «отзыв» встречается 0 раз). Автоматическая загрузка потребовала бы Selenium/Playwright, что выходит за рамки требований для непродвинутой группы. Два альтернативных сайта (zoon.ru, otzovik.com) заблокировали запросы. Поэтому данные собраны вручную с публично доступных страниц, каждая строка содержит ссылку на источник.

**Инструменты:** `pandas`, `matplotlib`

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

os.makedirs('../figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

print('Библиотеки загружены.')

## Шаг 1. Загружаем данные

In [ ]:
df = pd.read_csv('../data/raw/reviews_raw.csv')

print(f'Загружено строк: {len(df)}')
print(f'Столбцы: {list(df.columns)}')
print()
print(df.head())

## Шаг 2. Очистка данных

In [ ]:
# Удаляем строки без текста отзыва
df = df.dropna(subset=['review_text'])

# Удаляем строки с пустым текстом
df = df[df['review_text'].str.strip() != '']

# Удаляем полные дубликаты
df = df.drop_duplicates()

print(f'После очистки: {len(df)} отзывов')
print()
print('Отзывов по кафе:')
print(df['cafe_name'].value_counts())

## Шаг 3. Тематическая разметка по ключевым словам

Для каждого отзыва проверяем, упоминается ли каждая тема. Используем `str.contains()` — простой и прозрачный метод.

In [ ]:
# Каждый столбец = 1, если тема упоминается в отзыве, иначе 0

df['topic_taste'] = df['review_text'].str.contains(
    'вкус|вкусн|невкусн|напит|чай', case=False, na=False).astype(int)

df['topic_service'] = df['review_text'].str.contains(
    'персонал|бариста|обслужива|сервис|сотрудник|грубо|вежлив|отношени', case=False, na=False).astype(int)

df['topic_price'] = df['review_text'].str.contains(
    'цен|дорог|рублей|стоит|деньг|дешев', case=False, na=False).astype(int)

df['topic_toppings'] = df['review_text'].str.contains(
    'тапиок|шарик|боба|желе|топпинг|джусбол|бабл', case=False, na=False).astype(int)

df['topic_waiting'] = df['review_text'].str.contains(
    'ждат|очередь|долго|быстр', case=False, na=False).astype(int)

df['topic_atmosphere'] = df['review_text'].str.contains(
    'атмосфер|уютн|интерьер|оформлен|место|настроени', case=False, na=False).astype(int)

df['topic_assortment'] = df['review_text'].str.contains(
    'меню|ассортимент|выбор|позиц', case=False, na=False).astype(int)

topic_cols = [
    'topic_taste', 'topic_service', 'topic_price',
    'topic_toppings', 'topic_waiting', 'topic_atmosphere', 'topic_assortment'
]

print('Темы добавлены. Сумма упоминаний по каждой теме:')
print(df[topic_cols].sum().sort_values(ascending=False))

## Шаг 4. Сохраняем очищенный датасет

In [ ]:
df.to_csv('../data/processed/reviews_clean.csv', index=False, encoding='utf-8-sig')
print(f'Сохранено: data/processed/reviews_clean.csv ({len(df)} строк)')

---
# Графики и анализ

## График 1: Частота тем во всех отзывах

In [ ]:
# Читабельные названия тем для подписей на графике
topic_labels = {
    'topic_taste':      'Вкус напитка',
    'topic_service':    'Персонал / сервис',
    'topic_price':      'Цена',
    'topic_toppings':   'Тапиока / топпинги',
    'topic_waiting':    'Скорость / ожидание',
    'topic_atmosphere': 'Атмосфера / место',
    'topic_assortment': 'Меню / ассортимент'
}

counts_all = df[topic_cols].sum().sort_values()
counts_all.index = [topic_labels[t] for t in counts_all.index]

plt.figure(figsize=(10, 5))
plt.barh(counts_all.index, counts_all.values, color='steelblue')
plt.title('Частота тем во всех отзывах')
plt.xlabel('Количество отзывов')
plt.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig('../figures/topics_all.png', dpi=100)
plt.show()

## График 2: Частота тем в негативных отзывах (оценка ≤ 2)

In [ ]:
df_neg = df[df['rating'] <= 2]
print(f'Негативных отзывов (оценка ≤ 2): {len(df_neg)}')

counts_neg = df_neg[topic_cols].sum().sort_values()
counts_neg.index = [topic_labels[t] for t in counts_neg.index]

plt.figure(figsize=(10, 5))
plt.barh(counts_neg.index, counts_neg.values, color='tomato')
plt.title('Частота тем в негативных отзывах (оценка ≤ 2)')
plt.xlabel('Количество отзывов')
plt.grid(axis='x', alpha=0.4)
plt.tight_layout()
plt.savefig('../figures/topics_negative.png', dpi=100)
plt.show()

## График 3: Количество отзывов по кафе

In [ ]:
reviews_by_cafe = df['cafe_name'].value_counts()

plt.figure(figsize=(10, 5))
plt.bar(reviews_by_cafe.index, reviews_by_cafe.values, color='steelblue')
plt.title('Количество отзывов по кафе (выборка)')
plt.xlabel('Кафе')
plt.ylabel('Количество отзывов')
plt.xticks(rotation=15, ha='right')
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('../figures/reviews_by_cafe.png', dpi=100)
plt.show()

## График 4: Распределение оценок

In [ ]:
rating_counts = df['rating'].value_counts().sort_index()

plt.figure(figsize=(8, 5))
plt.bar(rating_counts.index, rating_counts.values, color='steelblue', width=0.6)
plt.title('Распределение оценок в отзывах')
plt.xlabel('Оценка (звёзды)')
plt.ylabel('Количество отзывов')
plt.xticks([1, 2, 3, 4, 5])
plt.grid(axis='y', alpha=0.4)
plt.tight_layout()
plt.savefig('../figures/rating_distribution.png', dpi=100)
plt.show()

---
## Выводы по анализу отзывов

На основе выборки из 44 отзывов пяти московских баблти-кафе можно сделать следующие выводы.

**Что важно покупателям (топ тем во всех отзывах):**
- **Вкус напитка** — самая часто упоминаемая тема. Покупатели ждут вкусного, хорошо приготовленного чая.
- **Персонал и сервис** — вторая по частоте тема. Дружелюбные и внимательные бариста значительно влияют на общее впечатление.
- **Тапиока и топпинги** — качество шариков упоминается особенно часто, причём нередко в негативном контексте (сырая, переваренная тапиока).

**Главные боли покупателей (топ тем в негативных отзывах):**
- **Качество тапиоки** — сырая или переваренная тапиока — главная причина низких оценок.
- **Отношение персонала** — грубость, равнодушие или невнимательность сотрудников.
- **Цена/качество** — покупатели замечают несоответствие цены (400–700 ₽) и качества продукта.

**Портрет целевой аудитории (выведен из косвенных признаков):**
Судя по стилю написания, тематике и частоте посещений, которые упоминаются в отзывах, основная аудитория — молодые люди, чувствительные к трендам, ценящие атмосферу и качество продукта, но также обращающие внимание на соотношение цены и качества.

**Важное ограничение:** выборка составляет 44 отзыва и не является статистически репрезентативной. Демографические характеристики (возраст, доход) не могут быть установлены напрямую из текстов отзывов. Выводы носят качественный, а не количественный характер.